# Convolution & Stencil Patterns

Implement 1D and 2D convolution on the GPU, from naive global memory approaches to optimized tiled implementations using shared memory and constant memory for filter weights.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-convolution)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: 1D Convolution

In [ ]:
!pip install numba

from numba import cuda
import numpy as np
import math
import time

# ------------------------------------------------------------------
# 1D Convolution: Naive GPU Implementation
# ------------------------------------------------------------------

@cuda.jit
def conv1d_naive(signal, weights, output, n, filter_size):
    """Naive 1D convolution: each thread loads all its inputs from global memory."""
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    radius = filter_size // 2
    if idx < n:
        result = 0.0
        for k in range(filter_size):
            input_idx = idx - radius + k
            if 0 <= input_idx < n:  # Zero-padding boundary
                result += signal[input_idx] * weights[k]
        output[idx] = result

# ------------------------------------------------------------------
# Test with different filters
# ------------------------------------------------------------------

# Create a test signal (noisy sine wave)
n = 100_000
t = np.linspace(0, 10 * np.pi, n, dtype=np.float32)
signal = np.sin(t) + 0.3 * np.random.randn(n).astype(np.float32)

# Different filter types
filters = {
    "Box blur (5)": np.array([0.2, 0.2, 0.2, 0.2, 0.2], dtype=np.float32),
    "Gaussian (5)": np.array([0.06, 0.24, 0.40, 0.24, 0.06], dtype=np.float32),
    "Derivative (3)": np.array([-1.0, 0.0, 1.0], dtype=np.float32),
    "Sharpen (3)": np.array([-1.0, 3.0, -1.0], dtype=np.float32),
}

TPB = 256
BPG = math.ceil(n / TPB)

d_signal = cuda.to_device(signal)

print("=== 1D Convolution with Different Filters ===")
print(f"Signal length: {n:,} samples\n")

for name, filt in filters.items():
    d_weights = cuda.to_device(filt)
    d_output = cuda.device_array(n, dtype=np.float32)

    # GPU computation
    conv1d_naive[BPG, TPB](d_signal, d_weights, d_output, n, len(filt))
    gpu_result = d_output.copy_to_host()

    # CPU reference (numpy)
    cpu_result = np.convolve(signal, filt, mode='same')

    # Verify
    max_err = np.max(np.abs(gpu_result - cpu_result))
    print(f"{name:<20} filter_size={len(filt)}, max_error={max_err:.2e} "
          f"{'PASS' if max_err < 1e-4 else 'FAIL'}")

# ------------------------------------------------------------------
# Performance benchmark: GPU naive vs NumPy
# ------------------------------------------------------------------
print("\n=== Performance: Naive GPU vs NumPy ===")

n_large = 2_000_000
signal_large = np.random.randn(n_large).astype(np.float32)
d_signal_large = cuda.to_device(signal_large)
d_output_large = cuda.device_array(n_large, dtype=np.float32)
BPG_large = math.ceil(n_large / TPB)

for fsize in [3, 7, 15, 31]:
    filt = np.ones(fsize, dtype=np.float32) / fsize  # Box filter
    d_filt = cuda.to_device(filt)

    # GPU timing
    for _ in range(5):
        conv1d_naive[BPG_large, TPB](d_signal_large, d_filt, d_output_large, n_large, fsize)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(20):
        conv1d_naive[BPG_large, TPB](d_signal_large, d_filt, d_output_large, n_large, fsize)
    cuda.synchronize()
    t_gpu = (time.perf_counter() - start) / 20 * 1000

    # NumPy timing
    start = time.perf_counter()
    for _ in range(5):
        _ = np.convolve(signal_large, filt, mode='same')
    t_np = (time.perf_counter() - start) / 5 * 1000

    reads_per_elem = fsize + 1  # filter + signal reads
    bw = n_large * reads_per_elem * 4 / (t_gpu / 1000) / 1e9
    print(f"  Filter size {fsize:>2}: GPU={t_gpu:.3f}ms, NumPy={t_np:.3f}ms, "
          f"Speedup={t_np/t_gpu:.1f}x, BW={bw:.0f} GB/s")

### Lesson example: 2D Convolution

In [ ]:
!pip install numba

from numba import cuda
import numpy as np
import math
import time

# ------------------------------------------------------------------
# 2D Convolution: Naive GPU Implementation
# ------------------------------------------------------------------

@cuda.jit
def conv2d_naive(image, filt, output, height, width, filt_size):
    """Naive 2D convolution -- each thread loads all filter inputs from global memory."""
    col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    radius = filt_size // 2

    if row < height and col < width:
        result = 0.0
        for fy in range(filt_size):
            for fx in range(filt_size):
                in_row = row - radius + fy
                in_col = col - radius + fx
                if 0 <= in_row < height and 0 <= in_col < width:
                    result += image[in_row, in_col] * filt[fy, fx]
        output[row, col] = result

# ------------------------------------------------------------------
# Create test image (synthetic pattern: checkerboard + gradient)
# ------------------------------------------------------------------
height, width = 1024, 1024
# Create a noisy gradient image
y_coords = np.linspace(0, 1, height, dtype=np.float32)
x_coords = np.linspace(0, 1, width, dtype=np.float32)
image = np.outer(y_coords, x_coords)  # Gradient
image += 0.1 * np.random.randn(height, width).astype(np.float32)  # Add noise
image = image.astype(np.float32)

print(f"Image size: {height} x {width} ({image.nbytes / 1e6:.1f} MB)")

# ------------------------------------------------------------------
# Apply different 2D filters
# ------------------------------------------------------------------
filters_2d = {
    "Box blur 3x3": np.ones((3, 3), dtype=np.float32) / 9.0,
    "Gaussian 3x3": np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=np.float32) / 16.0,
    "Sobel X": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32),
    "Sobel Y": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32),
    "Laplacian": np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=np.float32),
    "Sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32),
}

block = (32, 8)  # 256 threads; x=32 for full warp coalescing
grid = (math.ceil(width / block[0]), math.ceil(height / block[1]))

d_image = cuda.to_device(image)

print(f"\nBlock: {block}, Grid: {grid}")
print(f"Total threads: {grid[0]*block[0]*grid[1]*block[1]:,}")
print(f"\n{'Filter':<16} {'Size':<6} {'FLOPs/px':<10} {'Result range'}")
print("-" * 55)

for name, filt in filters_2d.items():
    fs = filt.shape[0]
    d_filt = cuda.to_device(filt)
    d_output = cuda.device_array((height, width), dtype=np.float32)

    conv2d_naive[grid, block](d_image, d_filt, d_output, height, width, fs)
    result = d_output.copy_to_host()

    flops_per_px = fs * fs * 2
    print(f"{name:<16} {fs}x{fs:<3} {flops_per_px:<10} "
          f"[{result.min():.3f}, {result.max():.3f}]")

# ------------------------------------------------------------------
# Sobel edge detection: combine Sx and Sy
# ------------------------------------------------------------------
print("\n=== Sobel Edge Detection ===")
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)

d_sx = cuda.to_device(sobel_x)
d_sy = cuda.to_device(sobel_y)
d_gx = cuda.device_array((height, width), dtype=np.float32)
d_gy = cuda.device_array((height, width), dtype=np.float32)

conv2d_naive[grid, block](d_image, d_sx, d_gx, height, width, 3)
conv2d_naive[grid, block](d_image, d_sy, d_gy, height, width, 3)

gx = d_gx.copy_to_host()
gy = d_gy.copy_to_host()
edge_magnitude = np.sqrt(gx**2 + gy**2)

print(f"Gradient X range: [{gx.min():.3f}, {gx.max():.3f}]")
print(f"Gradient Y range: [{gy.min():.3f}, {gy.max():.3f}]")
print(f"Edge magnitude range: [{edge_magnitude.min():.3f}, {edge_magnitude.max():.3f}]")
print(f"Strong edges (mag > 0.5): {np.sum(edge_magnitude > 0.5):,} pixels "
      f"({np.sum(edge_magnitude > 0.5) / (height * width) * 100:.1f}%)")

# ------------------------------------------------------------------
# Performance: filter size impact
# ------------------------------------------------------------------
print("\n=== Performance vs Filter Size ===")
print(f"{'Filter Size':<14} {'Time (ms)':<12} {'GFLOPS':<10} {'BW (GB/s)'}")
print("-" * 48)

for fs in [3, 5, 7, 9, 11]:
    filt = np.ones((fs, fs), dtype=np.float32) / (fs * fs)
    d_filt = cuda.to_device(filt)
    d_out = cuda.device_array((height, width), dtype=np.float32)

    for _ in range(5):
        conv2d_naive[grid, block](d_image, d_filt, d_out, height, width, fs)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(20):
        conv2d_naive[grid, block](d_image, d_filt, d_out, height, width, fs)
    cuda.synchronize()
    t = (time.perf_counter() - start) / 20 * 1000

    total_flops = height * width * fs * fs * 2
    total_bytes = height * width * (fs * fs + 1) * 4  # reads + write
    gflops = total_flops / (t / 1000) / 1e9
    bw = total_bytes / (t / 1000) / 1e9
    print(f"{fs}x{fs:<11} {t:<12.3f} {gflops:<10.1f} {bw:.1f}")

### Lesson example: Tiled Convolution with Shared Memory

In [ ]:
!pip install numba

from numba import cuda
import numba
import numpy as np
import math
import time

# ------------------------------------------------------------------
# Tiled 2D Convolution with Shared Memory
# ------------------------------------------------------------------

TILE_SIZE = 16
MAX_RADIUS = 7  # Support up to 15x15 filters
SHARED_DIM = TILE_SIZE + 2 * MAX_RADIUS  # 30

@cuda.jit
def conv2d_naive(image, filt, output, height, width, filt_size):
    """Naive: all reads from global memory."""
    col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    radius = filt_size // 2
    if row < height and col < width:
        result = 0.0
        for fy in range(filt_size):
            for fx in range(filt_size):
                iy = row - radius + fy
                ix = col - radius + fx
                if 0 <= iy < height and 0 <= ix < width:
                    result += image[iy, ix] * filt[fy, fx]
        output[row, col] = result

@cuda.jit
def conv2d_tiled(image, filt, output, height, width, filt_size):
    """Tiled convolution: cooperatively load input + halo into shared memory."""
    radius = filt_size // 2

    # Shared memory for tile + halo
    shared = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=numba.float32)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    # Output coordinates
    out_x = cuda.blockIdx.x * TILE_SIZE + tx
    out_y = cuda.blockIdx.y * TILE_SIZE + ty

    # Cooperatively load shared memory (each thread loads multiple elements)
    shared_w = TILE_SIZE + 2 * radius
    shared_h = TILE_SIZE + 2 * radius

    for sy in range(ty, shared_h, TILE_SIZE):
        for sx in range(tx, shared_w, TILE_SIZE):
            # Map shared memory coords to global coords
            gy = cuda.blockIdx.y * TILE_SIZE + sy - radius
            gx = cuda.blockIdx.x * TILE_SIZE + sx - radius
            if 0 <= gy < height and 0 <= gx < width:
                shared[sy, sx] = image[gy, gx]
            else:
                shared[sy, sx] = 0.0  # Zero padding

    cuda.syncthreads()  # All loads must complete before computing

    # Compute convolution from shared memory
    if out_y < height and out_x < width:
        result = 0.0
        for fy in range(filt_size):
            for fx in range(filt_size):
                result += shared[ty + fy, tx + fx] * filt[fy, fx]
        output[out_y, out_x] = result

# ------------------------------------------------------------------
# Benchmarking
# ------------------------------------------------------------------
def bench_2d(kernel, grid, block, args, warmup=10, iters=30):
    for _ in range(warmup):
        kernel[grid, block](*args)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        kernel[grid, block](*args)
    cuda.synchronize()
    return (time.perf_counter() - start) / iters * 1000

# Setup
height, width = 2048, 2048
image = np.random.randn(height, width).astype(np.float32)
d_image = cuda.to_device(image)
d_output_naive = cuda.device_array((height, width), dtype=np.float32)
d_output_tiled = cuda.device_array((height, width), dtype=np.float32)

# Naive grid/block
block_naive = (16, 16)
grid_naive = (math.ceil(width / 16), math.ceil(height / 16))

# Tiled grid/block
block_tiled = (TILE_SIZE, TILE_SIZE)
grid_tiled = (math.ceil(width / TILE_SIZE), math.ceil(height / TILE_SIZE))

print(f"Image: {height}x{width} ({height*width*4/1e6:.1f} MB)")
print(f"Naive block: {block_naive}, Tiled block: {block_tiled}")
print(f"\n{'Filter':<10} {'Naive (ms)':<14} {'Tiled (ms)':<14} {'Speedup':<10} {'Correct?'}")
print("-" * 60)

for fs in [3, 5, 7, 9, 11, 13]:
    if fs // 2 > MAX_RADIUS:
        continue
    filt = np.random.randn(fs, fs).astype(np.float32)
    filt /= filt.sum()  # Normalize
    d_filt = cuda.to_device(filt)

    t_naive = bench_2d(conv2d_naive, grid_naive, block_naive,
                       (d_image, d_filt, d_output_naive, height, width, fs))
    t_tiled = bench_2d(conv2d_tiled, grid_tiled, block_tiled,
                       (d_image, d_filt, d_output_tiled, height, width, fs))

    # Verify correctness
    out_n = d_output_naive.copy_to_host()
    out_t = d_output_tiled.copy_to_host()
    correct = np.allclose(out_n, out_t, atol=1e-4)

    speedup = t_naive / t_tiled
    print(f"{fs}x{fs:<7} {t_naive:<14.3f} {t_tiled:<14.3f} {speedup:<10.2f}x {correct}")

# ------------------------------------------------------------------
# Analysis: Why tiling helps more for larger filters
# ------------------------------------------------------------------
print("\n=== Why Tiling Helps More for Larger Filters ===")
print(f"{'Filter':<10} {'Global loads/px (Naive)':<25} {'Global loads/px (Tiled)':<25} {'Reduction'}")
print("-" * 72)
for fs in [3, 5, 7, 9, 11, 13]:
    radius = fs // 2
    naive_loads = fs * fs
    shared_w = TILE_SIZE + 2 * radius
    tiled_loads = (shared_w * shared_w) / (TILE_SIZE * TILE_SIZE)
    reduction = naive_loads / tiled_loads
    print(f"{fs}x{fs:<7} {naive_loads:<25} {tiled_loads:<25.1f} {reduction:.1f}x")

---

## Exercise: Sobel Edge Detection with Tiled 2D Convolution


In [ ]:
from numba import cuda
import numba
import numpy as np
import math
import time

TILE_SIZE = 16

# --- Naive 2D convolution (reference) ---
@cuda.jit
def conv2d_naive(image, filt, output, height, width, filt_size):
    col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    radius = filt_size // 2
    if row < height and col < width:
        result = 0.0
        for fy in range(filt_size):
            for fx in range(filt_size):
                iy = row - radius + fy
                ix = col - radius + fx
                if 0 <= iy < height and 0 <= ix < width:
                    result += image[iy, ix] * filt[fy, fx]
        output[row, col] = result

# --- TODO: Tiled 2D convolution with shared memory ---
@cuda.jit
def conv2d_tiled(image, filt, output, height, width, filt_size):
    # TODO: Declare shared memory of size (TILE_SIZE+2, TILE_SIZE+2) for 3x3 filter
    # TODO: Load tile + halo into shared memory cooperatively
    # TODO: cuda.syncthreads()
    # TODO: Compute convolution from shared memory
    pass

# --- Setup ---
height, width = 1024, 1024
image = np.random.randn(height, width).astype(np.float32)

sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)

# TODO: Apply both Sobel filters using naive and tiled kernels
# TODO: Compute edge magnitude sqrt(gx^2 + gy^2)
# TODO: Verify correctness
# TODO: Benchmark and report speedup

## Your tasks

Implement Sobel edge detection using an optimized tiled 2D convolution with shared memory.

**Requirements:**
1. Implement a tiled 2D convolution kernel that uses shared memory for the input tile + halo
2. Apply the Sobel X filter `[[-1,0,1],[-2,0,2],[-1,0,1]]` and Sobel Y filter `[[-1,-2,-1],[0,0,0],[1,2,1]]`
3. Compute edge magnitude as `sqrt(gx^2 + gy^2)` (can be done as a separate kernel or on CPU)
4. Compare performance of your tiled version against a naive version
5. Verify correctness by comparing tiled output against naive output
6. Test on at least a 1024x1024 image

**Hints:**
- Use a tile size of 16x16 with block size (16, 16)
- For a 3x3 Sobel filter (radius 1), shared memory needs to be 18x18
- Remember: `cuda.syncthreads()` between loading and computing!
- Use a loop to load the halo elements (threads at block edges load extra)
- Zero-padding for boundary handling is simplest
- Verify with: `np.allclose(tiled_result, naive_result, atol=1e-4)`

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-convolution) for the solution and discussion.